## Step 1: Configure Input Widgets

In [ ]:
# Remove existing widgets (safe to re-run)
dbutils.widgets.removeAll()
 
# Connection details
dbutils.widgets.text("connection_name", "", "1. Connection Name")
dbutils.widgets.text("mssql_host", "", "2. MSSQL Host (e.g. server.database.windows.net)")
dbutils.widgets.text("mssql_port", "1433", "3. MSSQL Port")
dbutils.widgets.text("mssql_database", "", "4. MSSQL Database Name")
dbutils.widgets.text("mssql_user", "", "5. MSSQL Username")
dbutils.widgets.text("mssql_password", "", "6. MSSQL Password")
 
# Foreign catalog details
dbutils.widgets.text("foreign_catalog_name", "", "7. Foreign Catalog Name (in Unity Catalog)")

## Step 2: Read & Validate Widget Values

In [ ]:
# Populate python variables for connection information
connection_name   = dbutils.widgets.get("connection_name").strip()
mssql_host        = dbutils.widgets.get("mssql_host").strip()
mssql_port        = dbutils.widgets.get("mssql_port").strip()
mssql_database    = dbutils.widgets.get("mssql_database").strip()
mssql_user        = dbutils.widgets.get("mssql_user").strip()
mssql_password    = dbutils.widgets.get("mssql_password").strip()
foreign_catalog   = dbutils.widgets.get("foreign_catalog_name").strip()

In [ ]:
# Validate all required fields are provided
required = {
    "connection_name":    connection_name,
    "mssql_host":         mssql_host,
    "mssql_port":         mssql_port,
    "mssql_database":     mssql_database,
    "mssql_user":         mssql_user,
    "mssql_password":     mssql_password,
    "foreign_catalog_name": foreign_catalog,
}
 
missing = [k for k, v in required.items() if not v]
 
if missing:
    raise ValueError(f"The following required widget values are missing or empty: {missing}")
 
print("✅ All widget values provided.")
print(f"   Connection Name   : {connection_name}")
print(f"   MSSQL Host        : {mssql_host}")
print(f"   MSSQL Port        : {mssql_port}")
print(f"   MSSQL Database    : {mssql_database}")
print(f"   MSSQL User        : {mssql_user}")
print(f"   MSSQL Password    : {'*' * len(mssql_password)}")
print(f"   Foreign Catalog   : {foreign_catalog}")

## Step 3: Create Unity Catalog Connection to MSSQL
Uses `CREATE CONNECTION IF NOT EXISTS` so the notebook is safe to re-run.

In [ ]:
# Connection creation
create_connection_sql = f"""
CREATE CONNECTION IF NOT EXISTS `{connection_name}`
TYPE sqlserver
OPTIONS (
  host      '{mssql_host}',
  port      '{mssql_port}',
  user      '{mssql_user}',
  password  '{mssql_password}',
  database  '{mssql_database}'
)
"""
 
spark.sql(create_connection_sql)
print(f"✅ Connection '{connection_name}' created (or already exists).")

## Step 4: Verify Connection Was Created

In [ ]:
SHOW CONNECTIONS;

## Step 5: Create Foreign Catalog
The foreign catalog maps a Unity Catalog catalog name to the MSSQL database via the connection created above.

In [ ]:
# Create a foreign CATALOG

create_catalog_sql = f"""
CREATE FOREIGN CATALOG IF NOT EXISTS `{foreign_catalog}`
USING CONNECTION `{connection_name}`
OPTIONS (database '{mssql_database}')
"""
 
spark.sql(create_catalog_sql)
print(f"✅ Foreign catalog '{foreign_catalog}' created (or already exists).")

## Step 6: Verify Foreign Catalog

In [ ]:
SHOW CATALOGS;

## Step 7: Browse Foreign Catalog Schemas (Optional Validation)

In [ ]:
# List all schemas in Foreign catalog

schemas_df = spark.sql(f"SHOW SCHEMAS IN `{foreign_catalog}`")
schemas_df.display()

In [ ]:
# Print few records in payments MSSQL source table

payments_df = spark.sql(f"SELECT * FROM {foreign_catalog}.dbo.payments LIMIT 10")
payments_df.display()

In [ ]:
# Print few records in orders MSSQL source table

orders_df = spark.sql(f"SELECT * FROM {foreign_catalog}.dbo.orders LIMIT 10")
orders_df.display()

In [ ]:
# Print few records in products MSSQL source table

products_df = spark.sql(f"SELECT * FROM {foreign_catalog}.dbo.products LIMIT 10")
products_df.display()